# 📦 PARTE 1 — Normalización del Dataset

## Celda 1 — Instalación e importaciones

In [3]:
!pip install -q pandas

import pandas as pd
import requests
import io
from google.colab import files, userdata

print("✅ Librerías listas")

✅ Librerías listas


## Celda 2 — Subir y explorar el dataset

In [6]:
print("📁 Seleccioná tu archivo CSV...")
uploaded = files.upload()

nombre_archivo = list(uploaded.keys())[0]
contenido = uploaded[nombre_archivo]

# Detecta encoding automáticamente
try:
    df = pd.read_csv(io.BytesIO(contenido), encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(io.BytesIO(contenido), encoding='latin1')
    print("ℹ️  Encoding: latin1")

print(f"\n✅ '{nombre_archivo}' cargado")
print(f"   📊 {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\n🔍 Vista previa:")
display(df.head(3))

print(f"\n📋 Tipos de datos:")
print(df.dtypes)

print(f"\n❓ Valores nulos:")
nulos = df.isnull().sum()
print(nulos[nulos > 0] if nulos.any() else "  ✅ Sin valores nulos")

📁 Seleccioná tu archivo CSV...


Saving sales_data_sample.csv to sales_data_sample.csv
ℹ️  Encoding: latin1

✅ 'sales_data_sample.csv' cargado
   📊 2,823 filas × 25 columnas

🔍 Vista previa:


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2/24/2003 0:00,Shipped,1,2,2003,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,5/7/2003 0:00,Shipped,2,5,2003,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,7/1/2003 0:00,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium



📋 Tipos de datos:
ORDERNUMBER           int64
QUANTITYORDERED       int64
PRICEEACH           float64
ORDERLINENUMBER       int64
SALES               float64
ORDERDATE            object
STATUS               object
QTR_ID                int64
MONTH_ID              int64
YEAR_ID               int64
PRODUCTLINE          object
MSRP                  int64
PRODUCTCODE          object
CUSTOMERNAME         object
PHONE                object
ADDRESSLINE1         object
ADDRESSLINE2         object
CITY                 object
STATE                object
POSTALCODE           object
COUNTRY              object
TERRITORY            object
CONTACTLASTNAME      object
CONTACTFIRSTNAME     object
DEALSIZE             object
dtype: object

❓ Valores nulos:
ADDRESSLINE2    2521
STATE           1486
POSTALCODE        76
TERRITORY       1074
dtype: int64


## Celda 3 — Normalización con Pandas

In [12]:
import numpy as np

df_original = df.copy()  # Copia de seguridad

# Reemplazar "Desconocido" por NaN
df = df.replace('Desconocido', np.nan)

# Convertir fechas
if 'ORDERDATE' in df.columns:
    df['ORDERDATE'] = pd.to_datetime(df['ORDERDATE'])
    print("✅ ORDERDATE convertida a datetime")

# Rellenar TODOS los nulos con 0
df = df.fillna(0)
print("✅ Todos los nulos rellenados con 0")

# Limpiar espacios
cols_texto = df.select_dtypes(include='object').columns
for col in cols_texto:
    df[col] = df[col].astype(str).str.strip()

# Eliminar duplicados
antes = len(df)
df = df.drop_duplicates()

eliminados = antes - len(df)

if eliminados > 0:
    print(f"✅ {eliminados} filas duplicadas eliminadas")
else:
    print("✅ Sin duplicados")

print(f"\n📊 Dataset normalizado: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"Nulos restantes: {df.isnull().sum().sum()}")

display(df.head(3))

✅ ORDERDATE convertida a datetime
✅ Todos los nulos rellenados con 0
✅ Sin duplicados

📊 Dataset normalizado: 2823 filas × 25 columnas
Nulos restantes: 0


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10107,30,95.70,2,2871.00,2003-02-24,Shipped,1,2,2003,...,897 Long Airport Avenue,0,NYC,NY,10022,USA,0,Yu,Kwai,Small
1,10121,34,81.35,5,2765.90,2003-05-07,Shipped,2,5,2003,...,59 rue de l'Abbaye,0,Reims,0,51100,France,EMEA,Henriot,Paul,Small
2,10134,41,94.74,2,3884.34,2003-07-01,Shipped,3,7,2003,...,27 rue du Colonel Pierre Avia,0,Paris,0,75508,France,EMEA,Da Cunha,Daniel,Medium


## Celda 4 — Exportar dataset normalizado

In [13]:
nombre_salida = nombre_archivo.replace('.csv', '_normalizado.csv')
df.to_csv(nombre_salida, index=False, encoding='utf-8')
files.download(nombre_salida)

print(f"✅ Dataset guardado: '{nombre_salida}'")
print(f"   {df.shape[0]:,} filas × {df.shape[1]} columnas | UTF-8")
print("⬇️  Descarga iniciada")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Dataset guardado: 'sales_data_sample_normalizado.csv'
   2,823 filas × 25 columnas | UTF-8
⬇️  Descarga iniciada


---
# 🤖 PARTE 2 — Chat con el Dataset usando Mistral AI

Modelo elegido: **`mistral-small-latest`**

**¿Por qué este modelo?**
- Es gratuito y no requiere tarjeta de crédito
- Tiene excelente comprensión de tablas y datos numéricos
- Responde en español sin problema
- Velocidad ideal para consultas interactivas

Otros modelos gratuitos disponibles:
| Modelo | Cuándo usarlo |
|---|---|
| `mistral-small-latest` ✅ | Consultas sobre datos, análisis, reportes |
| `open-mistral-7b` | Tareas simples, respuestas cortas |
| `open-mixtral-8x7b` | Análisis más complejos, más lento |

## Celda 5 — Conectar con Mistral AI

In [14]:
MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')
MODELO = 'mistral-small-latest'

if not MISTRAL_API_KEY:
    raise ValueError("⚠️ No se encontró MISTRAL_API_KEY en Secrets.")

def consultar_mistral(pregunta: str, contexto_datos: str) -> str:
    """Envía una pregunta a Mistral junto con el contexto del dataset."""
    prompt = f"""Sos un analista de datos experto. Tenés acceso a un dataset de ventas.

DATOS DEL DATASET (muestra representativa):
{contexto_datos}

Respondé la siguiente pregunta basándote SOLO en los datos proporcionados.
Sé claro, conciso y respondé en español.

PREGUNTA: {pregunta}"""

    r = requests.post(
        "https://api.mistral.ai/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {MISTRAL_API_KEY}",
            "Content-Type": "application/json"
        },
        json={
            "model": MODELO,
            "messages": [{"role": "user", "content": prompt}]
        }
    )
    if r.status_code != 200:
        raise Exception(f"Error API: {r.status_code} — {r.text}")
    return r.json()["choices"][0]["message"]["content"]

# Prepara el contexto del dataset para pasarle a Mistral
# Usamos estadísticas + muestra para no superar el límite de tokens
resumen_stats = df.describe(include='all').to_string()
muestra_datos = df.head(20).to_string(index=False)

contexto_datos = f"""
=== ESTADÍSTICAS GENERALES ===
{resumen_stats}

=== MUESTRA (primeras 20 filas) ===
{muestra_datos}

=== INFO ADICIONAL ===
Total filas: {df.shape[0]:,}
Columnas: {list(df.columns)}
Países únicos: {df['COUNTRY'].unique().tolist() if 'COUNTRY' in df.columns else 'N/A'}
Líneas de producto: {df['PRODUCTLINE'].unique().tolist() if 'PRODUCTLINE' in df.columns else 'N/A'}
Años disponibles: {sorted(df['YEAR_ID'].unique().tolist()) if 'YEAR_ID' in df.columns else 'N/A'}
"""

print(f"✅ Mistral listo — modelo: {MODELO}")
print(f"📊 Contexto preparado: {df.shape[0]:,} filas cargadas")

✅ Mistral listo — modelo: mistral-small-latest
📊 Contexto preparado: 2,823 filas cargadas


## Celda 6 — Hacé preguntas sobre tu dataset

Cambiá la variable `pregunta` y volvé a correr la celda para hacer consultas distintas.

In [15]:
# ── ESCRIBÍ TU PREGUNTA ACÁ ──────────────────────────────────────────
pregunta = "¿Cuál es el producto más vendido y en qué país se vende más?"
# ─────────────────────────────────────────────────────────────────────

print(f"❓ Pregunta: {pregunta}\n")
print("🤖 Consultando a Mistral AI...\n")

respuesta = consultar_mistral(pregunta, contexto_datos)

print("═" * 60)
print("💬 RESPUESTA DE MISTRAL AI")
print("═" * 60)
print(respuesta)

❓ Pregunta: ¿Cuál es el producto más vendido y en qué país se vende más?

🤖 Consultando a Mistral AI...

════════════════════════════════════════════════════════════
💬 RESPUESTA DE MISTRAL AI
════════════════════════════════════════════════════════════
**Producto más vendido:**
**Motorcycles** (Motocicletas) con un total de **967 unidades** (frecuencia en el dataset).

**País donde más se vende:**
**USA** (Estados Unidos) con **1,407 registros** (frecuencia en el dataset).

*Nota: Los datos muestran que "Motorcycles" es la línea de producto más frecuente en el dataset (967 veces), y USA es el país con mayor cantidad de registros (1,407). Esto no necesariamente indica volumen de ventas en unidades o ingresos, sino la presencia de registros en el dataset.*


## Celda 7 — Preguntas de ejemplo para explorar el dataset

Copiá cualquiera de estas en la celda anterior:

In [16]:
preguntas_ejemplo = [
    "¿Cuál fue el mes con mayores ventas totales?",
    "¿Qué línea de productos genera más ingresos?",
    "¿Cuáles son los 3 clientes que más compran?",
    "¿Cuál es el promedio de ventas por país?",
    "¿Hay diferencia de ventas entre los años disponibles?",
    "¿Qué tamaño de deal (DEALSIZE) es el más frecuente?",
    "Dame un resumen ejecutivo del rendimiento de ventas."
]

print("💡 Preguntas sugeridas para explorar el dataset:\n")
for i, p in enumerate(preguntas_ejemplo, 1):
    print(f"  {i}. {p}")

print("\n👆 Copiá cualquiera en la variable 'pregunta' de la Celda 6 y volvé a correrla.")

💡 Preguntas sugeridas para explorar el dataset:

  1. ¿Cuál fue el mes con mayores ventas totales?
  2. ¿Qué línea de productos genera más ingresos?
  3. ¿Cuáles son los 3 clientes que más compran?
  4. ¿Cuál es el promedio de ventas por país?
  5. ¿Hay diferencia de ventas entre los años disponibles?
  6. ¿Qué tamaño de deal (DEALSIZE) es el más frecuente?
  7. Dame un resumen ejecutivo del rendimiento de ventas.

👆 Copiá cualquiera en la variable 'pregunta' de la Celda 6 y volvé a correrla.
